# GAAE Training — DELCODE Whole-Brain (Schaefer 400)

This notebook trains the GAAE model on whole-brain correlation matrices (400 nodes, Schaefer atlas)
from `DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices`. It uses `GraphDatasetInMemoryFiltered` for whole-brain data.
It does a subject-level 60/20/20 train/val/test split via pre-generated CSVs.

In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to outputs/<experiment-id>/ when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name


In [ ]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path

import random
import numpy as np
import torch
from torch_geometric.loader import DataLoader

base_dir = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

print(f'Added to sys.path: {base_dir}')

from model.GAAE.models import GraphAttentionAutoencoderConditioned
from model.GAAE.dataset import GraphDatasetInMemoryFiltered
from model.GAAE.utils import knn_binary_adjacency_matrix_no_diag
from model.GAAE.train import train_model_with_val_notebook_train_loss

from common.seeding import set_seed, make_rng, make_torch_generator, seed_worker
from common.provenance import region_from_data_root


In [ ]:
# reproducibility seeding — must run before datasets, samplers, or models.
SEED = 100
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
import sys
if '/mnt/e/fyassine/ad-early-detection' not in sys.path:
    sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

WANDB_PROJECT = "ad-early-detection-whole-brain"
try:
    import wandb
    wandb.login()
except Exception:
    wandb = None
    print("wandb not available — logging disabled")

# ── Paths ───────────────────────────────────────────────
WB_DATA_ROOT  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
METADATA_DIR  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
SPLITS_DIR    = str(splits_dir('pretrain'))
TRAIN_CSV     = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV       = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV      = os.path.join(SPLITS_DIR, 'test.csv')

# Brain region / atlas parsed from the DATA directory name. Surfaced in the run
# name and stored in the run config so the input data is visible at a glance.
DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION    = DATA_INFO['region']
print(f"Input data: region={DATA_INFO['region']}  atlas={DATA_INFO['atlas']}  ({DATA_INFO['dataset_dir']})")

CHECKPOINT_SEARCH_DIRS = [
    str(base_dir / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
]
OUTPUT_DIR = str(base_dir / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── GAAE training hyper-params (loaded from configs/, inline fallback) ─────
CONFIG_PATH = base_dir / 'configs' / 'gaae_delcode_whole_brain.json'
TRAIN_CONFIG_PATH = CONFIG_PATH
_train_defaults = {
    "seed": 100, "batch_size": 64, "learning_rate": 0.001, "weight_decay": 0.001,
    "adj_loss_weight": 0.2, "epochs": 500, "early_stopping_patience": 25,
    "latent_dim": 64, "hidden_dim": 128, "num_heads": 2,
    "cond_dim": 2, "dropout": 0.3, "adjacency_k": 16, "num_workers": 8,
    "file_variant": "z_transformed",
}
if TRAIN_CONFIG_PATH.exists():
    with open(TRAIN_CONFIG_PATH) as _f:
        TRAIN_CONFIG = json.load(_f)
    print(f'Loaded training config from {TRAIN_CONFIG_PATH}')
else:
    TRAIN_CONFIG = _train_defaults
    print(f'Training config not found at {TRAIN_CONFIG_PATH} — using inline defaults.')

RANDOM_STATE            = TRAIN_CONFIG['seed']
set_seed(RANDOM_STATE)
BATCH_SIZE              = TRAIN_CONFIG['batch_size']
LEARNING_RATE           = TRAIN_CONFIG['learning_rate']
WEIGHT_DECAY            = TRAIN_CONFIG.get('weight_decay', 0.001)
ADJ_LOSS_WEIGHT         = TRAIN_CONFIG['adj_loss_weight']
EPOCHS                  = TRAIN_CONFIG['epochs']
EARLY_STOPPING_PATIENCE = TRAIN_CONFIG['early_stopping_patience']
LATENT_DIM              = TRAIN_CONFIG['latent_dim']
HIDDEN_DIM              = TRAIN_CONFIG.get('hidden_dim', 128)
NUM_HEADS               = TRAIN_CONFIG['num_heads']
COND_DIM                = TRAIN_CONFIG['cond_dim']
DROPOUT                 = TRAIN_CONFIG['dropout']
ADJACENCY_K             = TRAIN_CONFIG['adjacency_k']
ADJACENCY_ARGS          = {'k': ADJACENCY_K}
NUM_WORKERS             = TRAIN_CONFIG['num_workers']
FILE_VARIANT            = TRAIN_CONFIG.get('file_variant', 'z_transformed')

print('Config set.')

In [ ]:
## Train vs Load Checkpoint

from pathlib import Path

_candidates = sorted(
    [
        d
        for search in CHECKPOINT_SEARCH_DIRS
        if Path(search).is_dir()
        for d in Path(search).iterdir()
        if d.is_dir() and (d / f"model_{d.name}.pth").exists()
    ],
    key=lambda d: d.name,
)

if _candidates:
    print("  0: Train a new model (default)")
    for i, d in enumerate(_candidates, start=1):
        print(f"  {i}: {d.name}")
    if GAAE_CHECKPOINT_PATH is not None:
        _idx_str = ''   # runner supplied an explicit checkpoint (handled below)
    elif RUN_DIR is not None:
        _idx_str = '0'  # headless default: train a new model
    else:
        _idx_str = input("Select [0=train new / 1,2,...=load existing, Enter=train new]: ").strip()
    _idx = int(_idx_str) if _idx_str.isdigit() else 0
else:
    print("No existing checkpoints found — will train a new model.")
    _idx = 0

if 1 <= _idx <= len(_candidates):
    _run_dir = _candidates[_idx - 1]
    USE_EXISTING_CHECKPOINT = True
    SELECTED_RUN_NAME = _run_dir.name
    SELECTED_RUN_DIR = _run_dir
    GAAE_CHECKPOINT_PATH = str(_run_dir / f"model_{_run_dir.name}.pth")
    print(f"Will load: {GAAE_CHECKPOINT_PATH}")
else:
    USE_EXISTING_CHECKPOINT = False
    GAAE_CHECKPOINT_PATH = None
    print("Will train a new model.")

## Dataset

In [ ]:

train_dataset = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=ADJACENCY_ARGS,
    filter_csv_path=TRAIN_CSV,
    patient_info_path=TRAIN_CSV,
    separator=",",
    file_variant=FILE_VARIANT,
)

val_dataset = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=ADJACENCY_ARGS,
    filter_csv_path=VAL_CSV,
    patient_info_path=VAL_CSV,
    separator=",",
    file_variant=FILE_VARIANT,
)

test_dataset = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=ADJACENCY_ARGS,
    filter_csv_path=TEST_CSV,
    patient_info_path=TEST_CSV,
    separator=",",
    file_variant=FILE_VARIANT,
)

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"Feature shape: {train_dataset[0].x.shape}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=torch_gen,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=torch_gen,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    pin_memory=True if torch.cuda.is_available() else False
)

dataset_info = {
    "dataset_name": "Whole-brain Schaefer 400-node (60/20/20 subject-level split)",
    "kNN_param": ADJACENCY_ARGS['k'],
    "correlation_type": FILE_VARIANT,
    "num_features": train_dataset[0].x.size(1),
    "train_dataset_size": len(train_dataset),
    "val_dataset_size": len(val_dataset),
    "test_dataset_size": len(test_dataset),
    "batch_size": BATCH_SIZE
}
print(f"Dataset info: {json.dumps(dataset_info, indent=2)}")

## Model

In [ ]:
in_features = train_dataset[0].x.size(1)
hidden_dim = HIDDEN_DIM

In [ ]:
model = GraphAttentionAutoencoderConditioned(
    in_features=in_features,
    hidden_dim=hidden_dim,
    out_features=LATENT_DIM,
    cond_dim=COND_DIM,
    num_heads=NUM_HEADS,
    dropout=DROPOUT
).to(device)

model_config = {
    "model_type": model.__class__.__name__,
    "in_features": in_features,
    "hidden_dim": hidden_dim,
    "latent_dim": LATENT_DIM,
    "attention_heads": NUM_HEADS,
    "device": device.type,
    "dropout": DROPOUT
}

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"Model: {model.__class__.__name__}")
print(f"  in_features={in_features}, hidden_dim={hidden_dim}, out_features={LATENT_DIM}")
print(f"  num_heads={NUM_HEADS}, dropout={DROPOUT}")
print(f"  optimizer=AdamW  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}")

In [ ]:
CHECKPOINT_SEARCH_DIRS = [
    str(base_dir / "notebooks" / "checkpoints" / "checkpoints_gaae_whole_brain"),
]

if USE_EXISTING_CHECKPOINT:
    ckpt = torch.load(GAAE_CHECKPOINT_PATH, map_location=device, weights_only=False)
    if isinstance(ckpt, torch.nn.Module):
        model = ckpt.to(device)
    elif isinstance(ckpt, dict):
        state = ckpt.get("model_state_dict", ckpt)
        model.load_state_dict(state)
        model = model.to(device)
    else:
        raise TypeError("Unsupported checkpoint format. Expected nn.Module or state dict.")

    best_model = model
    run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"checkpoint_{SELECTED_RUN_NAME}_{run_timestamp}"
    print(f"Loaded checkpoint from {GAAE_CHECKPOINT_PATH}")
    print("Training skipped because USE_EXISTING_CHECKPOINT=True")

    history_file = SELECTED_RUN_DIR / "history.json"
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
        print(f"Loaded training history from {history_file}")
    else:
        history = {"train_loss": [], "val_loss": []}
        print(f"No saved training history at {history_file} — loss curve will be empty for this checkpoint.")
else:
    best_state_dict, history = train_model_with_val_notebook_train_loss(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        model_config=model_config,
        adj_loss_weight=ADJ_LOSS_WEIGHT,
        epochs=EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        dataset_info=dataset_info,
        project_name=WANDB_PROJECT
    )

    # train_model_with_val_notebook_train_loss returns the best-val-loss
    # weights as a state_dict, not a module. Load it back into `model` so
    # both `model` and `best_model` carry the best (not final-epoch) weights.
    model.load_state_dict(best_state_dict)
    best_model = model

    run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    try:
        wandb_run_name = wandb.run.name if wandb and wandb.run and wandb.run.name else "run"
    except Exception:
        wandb_run_name = "run"
    wandb_run_name = str(wandb_run_name).replace(" ", "-")
    run_name = f"{wandb_run_name}_{run_timestamp}"

    checkpoint_root = base_dir / "notebooks" / "checkpoints" / "checkpoints_gaae_whole_brain"
    os.makedirs(checkpoint_root, exist_ok=True)
    run_artifact_dir = checkpoint_root / run_name
    os.makedirs(run_artifact_dir, exist_ok=True)

    model_file = run_artifact_dir / f"model_{run_name}.pth"
    torch.save(best_model, str(model_file))
    print(f"Saved best model to {model_file}")

    config_to_save = {
        "run_name": run_name,
        "wandb_run_name": wandb_run_name,
        "timestamp": run_timestamp,
        "dataset_info": dataset_info,
        "model_config": model_config,
        "training_config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "adj_loss_weight": ADJ_LOSS_WEIGHT,
            "epochs": EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE
        }
    }

    def json_serial(obj):
        if isinstance(obj, (datetime, torch.device)):
            return str(obj)
        raise TypeError(f"Type {type(obj)} not serializable")

    config_file = run_artifact_dir / "run_config.json"
    with open(config_file, "w") as f:
        json.dump(config_to_save, f, indent=4, default=json_serial)
    print(f"Saved run configuration to {config_file}")

    history_file = run_artifact_dir / "history.json"
    with open(history_file, "w") as f:
        json.dump(history, f, indent=4)
    print(f"Saved training history to {history_file}")

In [ ]:
import matplotlib.pyplot as plt

if history['train_loss']:
    plt.figure(figsize=(10, 6))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss (Whole-Brain 400-node)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No training-loss history available for this run — skipping plot.")

## Loss

In [ ]:
import pandas as pd
from model.GAAE.evaluation import compute_errors_for_dataset, plot_cohort_errors

train_split_df = pd.read_csv(TRAIN_CSV)
val_split_df   = pd.read_csv(VAL_CSV)

assert "Pseudonym" in train_split_df.columns and "diagnosis" in train_split_df.columns
assert "Pseudonym" in val_split_df.columns   and "diagnosis" in val_split_df.columns

allowed_cohorts = {"ad", "converter", "healthy", "mci"}

combined_split_df = pd.concat([train_split_df, val_split_df], ignore_index=True)
subject_cohort_map = (
    combined_split_df[["Pseudonym", "diagnosis"]]
    .drop_duplicates(subset=["Pseudonym"], keep="first")
    .assign(
        Pseudonym=lambda d: d["Pseudonym"].astype(str).str.strip(),
        diagnosis=lambda d: d["diagnosis"].astype(str).str.lower().str.strip(),
    )
    .set_index("Pseudonym")["diagnosis"]
    .to_dict()
)

train_errors_df = compute_errors_for_dataset(
    train_dataset, "Train", model, device, subject_cohort_map, ADJ_LOSS_WEIGHT,
    allowed_cohorts=allowed_cohorts,
)
val_errors_df = compute_errors_for_dataset(
    val_dataset, "Validation", model, device, subject_cohort_map, ADJ_LOSS_WEIGHT,
    allowed_cohorts=allowed_cohorts,
)

cohort_order = [
    c for c in ["ad", "converter", "healthy", "mci"]
    if c in set(train_errors_df["Cohort"]) | set(val_errors_df["Cohort"])
]

_run_name = globals().get("run_name", globals().get("SELECTED_RUN_NAME", "run"))
for _split_name, _split_df, _palette in [
    ("Train",      train_errors_df, "Blues"),
    ("Validation", val_errors_df,   "Greens"),
]:
    plot_cohort_errors(_split_name, _split_df, cohort_order, _palette, WANDB_PROJECT, _run_name)

print("Train cohort counts:")
print(train_errors_df.groupby("Cohort").size().sort_index())
print("\nValidation cohort counts:")
print(val_errors_df.groupby("Cohort").size().sort_index())

In [ ]:
try:
    wandb.finish()
except Exception:
    pass

## Robustness Evaluation (Threshold from Reconstruction Error)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from CLASSIFIER.common.robustness import perturb_graph
from model.GAAE.losses import compute_sample_reconstruction_error
from model.GAAE.utils import knn_binary_adjacency_matrix_no_diag
from model.GAAE.evaluation import (
    compute_errors_for_dataset,
    compute_one_vs_rest_thresholds,
    is_cohort_positive,
    plot_robustness_sweep,
)

# ── Resolve eval model ────────────────────────────────────────────────────
if 'best_model' in globals() and best_model is not None:
    eval_model = best_model
elif 'model' in globals() and model is not None:
    eval_model = model
else:
    raise RuntimeError('No trained model found. Run training or checkpoint cells first.')
eval_model = eval_model.to(device)
eval_model.eval()

if 'val_errors_df' not in globals():
    raise RuntimeError('val_errors_df missing — run the cohort error cell first.')

# ── Build full cohort map (train + val + test) ────────────────────────────
subject_cohort_map_all = dict(subject_cohort_map)
subject_cohort_map_all.update(
    pd.read_csv(TEST_CSV)[['Pseudonym', 'diagnosis']]
    .dropna()
    .drop_duplicates(subset=['Pseudonym'])
    .assign(
        Pseudonym=lambda d: d['Pseudonym'].astype(str),
        diagnosis=lambda d: d['diagnosis'].astype(str).str.lower().str.strip(),
    )
    .set_index('Pseudonym')['diagnosis']
    .to_dict()
)

# ── Val-derived thresholds ────────────────────────────────────────────────
cohorts_to_analyze = ['healthy', 'ad', 'mci', 'converter']
cohort_thresholds  = compute_one_vs_rest_thresholds(val_errors_df, cohorts_to_analyze)

threshold_rows = [
    {'Cohort': c, 'Direction': cohort_thresholds[c]['direction'],
     'AUC': cohort_thresholds[c]['auc'],
     'ThresholdError': cohort_thresholds[c]['threshold_error']}
    for c in cohorts_to_analyze
]
print('Validation one-vs-rest thresholds:')
print(pd.DataFrame(threshold_rows).to_string(index=False))

# ── Test errors + cohort predictions ─────────────────────────────────────
test_errors_eval_df = compute_errors_for_dataset(
    test_dataset, 'Test', eval_model, device, subject_cohort_map_all, ADJ_LOSS_WEIGHT,
)
for c in cohorts_to_analyze:
    test_errors_eval_df[f'Pred_{c}'] = test_errors_eval_df['Total Error'].apply(
        lambda e: is_cohort_positive(float(e), cohort_thresholds[c])
    )

# ── Top-k selection ───────────────────────────────────────────────────────
top_k = 5
selected_frames = []
for cohort_name in cohorts_to_analyze:
    th = cohort_thresholds[cohort_name]
    if np.isnan(th['threshold_error']):
        print(f'Skipping {cohort_name}: threshold is NaN.')
        continue
    candidates = test_errors_eval_df[
        (test_errors_eval_df['Cohort'] == cohort_name) &
        (test_errors_eval_df[f'Pred_{cohort_name}'] == 1)
    ].copy()
    candidates['SelectionMargin'] = (
        candidates['Total Error'] - th['threshold_error'] if th['direction'] == 'high'
        else th['threshold_error'] - candidates['Total Error']
    )
    if candidates.empty:
        print(f'No eligible samples for cohort={cohort_name}.')
        continue
    top = candidates.sort_values('SelectionMargin', ascending=False).head(top_k).copy()
    top['SelectionCohort'] = cohort_name
    selected_frames.append(top)

if not selected_frames:
    raise RuntimeError('No selected samples found for any cohort.')
selected_samples_df = pd.concat(selected_frames, ignore_index=True)
print(f'\nSelected samples per cohort:')
print(selected_samples_df.groupby('SelectionCohort').size().sort_index().to_string())

# ── Robustness sweep ──────────────────────────────────────────────────────
noise_levels  = [0.00, 0.05, 0.10, 0.20, 0.30]
n_trials      = 10
noise_methods = ['feature_noise', 'matrix_noise_rebuild', 'edge_perturbation', 'conditioning_noise']

records = []
for _, row in selected_samples_df.iterrows():
    ds_idx      = int(row['DatasetIndex'])
    base_sample = test_dataset[ds_idx]
    patient_id  = str(row['PatientID'])
    sel_cohort  = str(row['SelectionCohort'])
    cohort_thr  = cohort_thresholds[sel_cohort]

    for method in noise_methods:
        for noise_level in noise_levels:
            for trial in range(n_trials):
                if noise_level > 0 and method != 'none':
                    if method == 'matrix_noise_rebuild':
                        d = perturb_graph(base_sample, 'feature_noise', noise_level, rng=rng)
                        adj_bin = knn_binary_adjacency_matrix_no_diag(d.x.cpu().numpy(), **ADJACENCY_ARGS)
                        src, dst = np.where(adj_bin > 0)
                        d.edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
                        d.edge_attr  = torch.ones(d.edge_index.size(1), dtype=torch.float32)
                    else:
                        d = perturb_graph(base_sample, method, noise_level, rng=rng)
                else:
                    d = base_sample

                _, _, total_err = compute_sample_reconstruction_error(
                    d, eval_model, device, ADJ_LOSS_WEIGHT
                )
                records.append({
                    'SelectionCohort': sel_cohort, 'PatientID': patient_id,
                    'DatasetIndex': ds_idx, 'Method': method,
                    'NoiseLevel': noise_level, 'NoiseLevelPercent': noise_level * 100.0,
                    'Trial': trial, 'Total Error': float(total_err),
                    'CohortStable': is_cohort_positive(float(total_err), cohort_thr),
                })

robustness_df = pd.DataFrame(records)
summary_df = (
    robustness_df
    .groupby(['SelectionCohort', 'Method', 'NoiseLevel', 'NoiseLevelPercent'], as_index=False)
    .agg(MeanTotalError=('Total Error', 'mean'), StdTotalError=('Total Error', 'std'),
         CohortStabilityRate=('CohortStable', 'mean'))
)
print('\nCohort stability summary:')
print(
    summary_df[['SelectionCohort', 'Method', 'NoiseLevelPercent', 'CohortStabilityRate']]
    .sort_values(['SelectionCohort', 'Method', 'NoiseLevelPercent'])
    .to_string(index=False)
)

plot_robustness_sweep(summary_df, cohorts_to_analyze, cohort_thresholds, noise_methods)

# ── Save artifacts ────────────────────────────────────────────────────────
eval_base_dir = Path(run_artifact_dir) if 'run_artifact_dir' in globals() else Path.cwd()
eval_dir = eval_base_dir / f'robustness_eval_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}'
eval_dir.mkdir(parents=True, exist_ok=True)

selected_samples_df.to_csv(eval_dir / 'selected_top5_by_cohort.csv',     index=False)
summary_df.to_csv(         eval_dir / 'robustness_summary_by_cohort.csv', index=False)
robustness_df.to_csv(      eval_dir / 'robustness_details_by_cohort.csv', index=False)
pd.DataFrame(threshold_rows).to_csv(eval_dir / 'cohort_thresholds.csv',   index=False)

with open(eval_dir / 'robustness_meta.json', 'w') as f:
    json.dump({
        'threshold_type': 'one_vs_rest_per_cohort_from_validation_reconstruction_error',
        'cohort_thresholds': cohort_thresholds,
        'noise_levels': noise_levels,
        'noise_levels_percent': [x * 100.0 for x in noise_levels],
        'n_trials': n_trials, 'methods': noise_methods,
        'cohorts_to_analyze': cohorts_to_analyze, 'top_k_per_cohort': top_k,
        'selected_count_by_cohort': selected_samples_df.groupby('SelectionCohort').size().to_dict(),
    }, f, indent=2)
print(f'Saved evaluation artifacts to: {eval_dir}')